In [1]:
import os
import csv
from pathlib import Path
import crowdwalk

In [ ]:
# DON'T EDIT THIS CELL

planned_route_list = {
                "Door1Left": ["DOOR_1F_1", "ROOT_2F_L06", "ROOT_2F_L01", "ROOT_2F_L02"],
                "Door1Right": ["DOOR_1F_6", "ROOT_2F_R06", "ROOT_2F_R01", "ROOT_2F_R02"],
                "Door2Left": ["DOOR_1F_2", "ROOT_2F_L06", "ROOT_2F_L01", "ROOT_2F_L02"],
                "Door2Right": ["DOOR_1F_5", "ROOT_2F_R01", "ROOT_2F_R02"],
                "Door3OuterLeft": ["DOOR_1F_3_1", "ROOT_2F_L01", "ROOT_2F_L02"],
                "Door3InnerLeft": ["DOOR_1F_3_2", "ROOT_2F_L05", "ROOT_2F_L01", "ROOT_2F_L02"],
                "Door3InnerRight": ["DOOR_1F_4_1", "ROOT_2F_R05", "ROOT_2F_R01", "ROOT_2F_R02"],
                "Door3OuterRight": ["DOOR_1F_4_2", "ROOT_2F_R01", "ROOT_2F_R02"]
}

seat_num = 0
seat_group_num = 0
seat_list = []
with open('scenario/shinkoku/seat.csv') as f:
    reader = csv.reader(f)
    is_first = True
    for row in reader:
        if is_first:
            casted_row = row
            is_first = False
        else:
            casted_row = [int(v) for v in row]
            seat_num += sum(casted_row[1:])
            seat_group_num += len(casted_row) - 1
        seat_list.append(casted_row)

def check_agent_route_list(agent_route_list):
    total_count = 0
    created_seat_set = set()
    for rule in agent_route_list:
        total_count += rule[6]
        created_seat_set.add(rule[1])
    if seat_num != total_count:
        raise RuntimeError(f'Invalied seats number: total seats should be {seat_num}. The created number is {total_count}.')
    if seat_group_num != len(created_seat_set):
        raise RuntimeError(f'Insufficient seats creation: whole seats should be created.')
    


def write_generation_file(agent_route_list):
    check_agent_route_list(agent_route_list)
    os.makedirs('work', exist_ok=True)
    with open('work/gen.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(agent_route_list)


def create_agent_rule(seat_col, seat_row, planned_route):
    rule = "TIMEEVERY"
    start_time = "18:00:00"
    duration = 1
    speed_model = "PLAIN"
    goal = "EXIT"
    start_place = "SEAT_1F_" + seat_list[0][seat_col] + str(seat_row).zfill(2)
    total = seat_list[seat_row][seat_col]

    line = [rule,
        start_place,
        start_time, start_time,
        duration, duration,
        total,
        speed_model,
        goal]

    line.extend(planned_route)
    return line

# DON'T EDIT THIS CELL

In [ ]:
def simulation(door1, door2, use_gui=False):
    door3 = 22 - door1 - door2
    
    print(f'Simulation Setting: Door1={door1}, Door2={door2}, Door3={door3}')
    
    if door3 < 0:
        raise ValueError(f"Invalid inputs: 22 - door1({door1}) - door2({door2}) = door3({door3}). The door3 should be not less than 0.")

    list = []

    # Generate agents that are exit from door 1
    for row in range(1, (door1 + 1)):
        for column in range(1, (3 + 1)):
            list.append(create_agent_rule(column, row, planned_route_list["Door1Left"]))
        for column in range(4, (6 + 1)):
            list.append(create_agent_rule(column, row, planned_route_list["Door1Right"]))

    # Generate agents that are exit from door 2
    for row in range((door1 + 1), (door1 + door2 + 1)):
        for column in range(1, (3 + 1)):
            list.append(create_agent_rule(column, row, planned_route_list["Door2Left"]))
        for column in range(4, (6 + 1)):
            list.append(create_agent_rule(column, row, planned_route_list["Door2Right"]))

    # Generate agents that are exit from door 3
    for row in range((door1 + door2 + 1), (door1 + door2 + door3 + 1)):
        for column in range(1, (2 + 1)):
            list.append(create_agent_rule(column, row, planned_route_list["Door3OuterLeft"]))
        for column in range(3, (3 + 1)):
            list.append(create_agent_rule(column, row, planned_route_list["Door3InnerLeft"]))
        for column in range(4, (4 + 1)):
            list.append(create_agent_rule(column, row, planned_route_list["Door3InnerRight"]))
        for column in range(5, (6 + 1)):
            list.append(create_agent_rule(column, row, planned_route_list["Door3OuterRight"]))

    # Create a generation file to workspace. The generation file defines route of people in simulation.
    write_generation_file(list)
    # The created file, gen.csv, will be referenced and Crowdwalk returns elapsed time.
    result = crowdwalk.run(use_gui=use_gui)

    return result

In [18]:
def simulation_by_rows(door1_rows, door2_rows, door3_rows, use_gui=False):
    print(f'Simulation Setting: Door1={door1_rows}, Door2={door2_rows}, Door3={door3_rows}')

    all_rows = list(door1_rows) + list(door2_rows) + list(door3_rows)
    if sorted(all_rows) != list(range(1, 23)):
        raise ValueError('door1_rows, door2_rows, door3_rows should contain each row number from 1 to 22 exactly once.')

    agent_route_list = []

    for row in door1_rows:
        for column in range(1, (3 + 1)):
            agent_route_list.append(create_agent_rule(column, row, planned_route_list["Door1Left"]))
        for column in range(4, (6 + 1)):
            agent_route_list.append(create_agent_rule(column, row, planned_route_list["Door1Right"]))

    for row in door2_rows:
        for column in range(1, (3 + 1)):
            agent_route_list.append(create_agent_rule(column, row, planned_route_list["Door2Left"]))
        for column in range(4, (6 + 1)):
            agent_route_list.append(create_agent_rule(column, row, planned_route_list["Door2Right"]))

    for row in door3_rows:
        for column in range(1, (2 + 1)):
            agent_route_list.append(create_agent_rule(column, row, planned_route_list["Door3OuterLeft"]))
        for column in range(3, (3 + 1)):
            agent_route_list.append(create_agent_rule(column, row, planned_route_list["Door3InnerLeft"]))
        for column in range(4, (4 + 1)):
            agent_route_list.append(create_agent_rule(column, row, planned_route_list["Door3InnerRight"]))
        for column in range(5, (6 + 1)):
            agent_route_list.append(create_agent_rule(column, row, planned_route_list["Door3OuterRight"]))

    write_generation_file(agent_route_list)
    result = crowdwalk.run(use_gui=use_gui)

    return result



In [27]:
result = simulation(door1=12, door2=0, use_gui=False)
print(f'Elapsed Time: {result}')

Simulation Setting: Door1=12, Door2=0, Door3=10
java -Dfile.encoding=UTF-8  -Djdk.gtk.version=2 -jar CrowdWalk-master/crowdwalk/build/libs/crowdwalk.jar -l Error -c scenario/shinkoku/prop.json
Elapsed Time: 328


In [ ]:
min_time = 100000
min_door1 = 0
min_door2 = 0
for door1 in range(23):
    for door2 in range(23 - door1):
        result = simulation(door1,door2,use_gui=False)
        if result < min_time:
            min_time = result
            min_door1 = door1
            min_door2 = door2

print(f'Elapsed Time: {min_time} door1: {min_door1} door2: {min_door2}')

Simulation Setting: Door1=0, Door2=0, Door3=22
java -Dfile.encoding=UTF-8  -Djdk.gtk.version=2 -jar CrowdWalk-master/crowdwalk/build/libs/crowdwalk.jar -l Error -c scenario/shinkoku/prop.json
Simulation Setting: Door1=0, Door2=1, Door3=21
java -Dfile.encoding=UTF-8  -Djdk.gtk.version=2 -jar CrowdWalk-master/crowdwalk/build/libs/crowdwalk.jar -l Error -c scenario/shinkoku/prop.json
Simulation Setting: Door1=0, Door2=2, Door3=20
java -Dfile.encoding=UTF-8  -Djdk.gtk.version=2 -jar CrowdWalk-master/crowdwalk/build/libs/crowdwalk.jar -l Error -c scenario/shinkoku/prop.json
Simulation Setting: Door1=0, Door2=3, Door3=19
java -Dfile.encoding=UTF-8  -Djdk.gtk.version=2 -jar CrowdWalk-master/crowdwalk/build/libs/crowdwalk.jar -l Error -c scenario/shinkoku/prop.json
Simulation Setting: Door1=0, Door2=4, Door3=18
java -Dfile.encoding=UTF-8  -Djdk.gtk.version=2 -jar CrowdWalk-master/crowdwalk/build/libs/crowdwalk.jar -l Error -c scenario/shinkoku/prop.json
Simulation Setting: Door1=0, Door2=5, Do